# Elastic AI: agentes, workflows e busca inteligente

Workshop TDC 2026 · **Alex Salgado** · Developer Advocate @ Elastic · [@alexsalgadoprof](https://github.com/alexsalgadoprof)

## Como usar este notebook no Google Colab

1. **Crie seu trial Elastic Cloud Serverless** em [cloud.elastic.co](https://cloud.elastic.co)
2. **Obtenha uma API key da Jina AI** em [jina.ai](https://jina.ai)
3. **Configure os Secrets do Colab** (🔑 na barra lateral esquerda):
   - `MY_NAME` — seu nome em minúsculas, sem espaço (ex: `alex`)
   - `MY_BIO` — descrição curta do seu perfil
   - `MY_ES_URL` — URL do Elasticsearch (ex: `https://xxx.es.us-central1.gcp.elastic.cloud:443`)
   - `MY_KIBANA_URL` — URL do Kibana (ex: `https://xxx.kb.us-central1.gcp.elastic.cloud:443`)
   - `MY_API_KEY` — API key do seu cluster
   - `JINA_API_KEY` — API key da Jina AI
4. **Faça upload do seu CV** (PDF do LinkedIn) quando solicitado
5. **Execute célula por célula**, acompanhando o passo a passo do slide

> 💡 **Alternativa aos Secrets:** você também pode fazer upload de um arquivo `.env` com as variáveis. O notebook detecta automaticamente qual método usar.

### Estrutura do notebook

| Fase | Passos | Objetivo |
|------|--------|----------|
| **Fase 1 — Setup do meu cluster** | 1 a 4 | Validar conectividade, configurar Jina, criar índice, ingerir CV, testar busca semântica |
| **Fase 2 — Meu agente Avatar** | 5 a 7 | Criar tool, criar agente, testar chat local, publicar no diretório |
| **Fase 3 — Conversando com a rede** | 8 a 10 | Descobrir colegas, invocar cross-cluster, orquestrar respostas combinadas |

A Fase 3 só funciona depois que pelo menos um colega publicou no diretório.


---
## Setup


In [21]:
%pip install -q httpx python-dotenv rich nest_asyncio


In [22]:
import asyncio
import json
import os
from pathlib import Path

import httpx
import nest_asyncio
from dotenv import load_dotenv
from rich.console import Console
from rich.json import JSON
from rich.panel import Panel
from rich.pretty import Pretty

nest_asyncio.apply()

console = Console()

# ── Detecta ambiente ──
try:
    from google.colab import userdata as colab_secrets
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Pasta do workshop no Drive ──
WORKSHOP_DIR = "/content/drive/MyDrive/workshop-tdc-2026"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(WORKSHOP_DIR, exist_ok=True)
    os.chdir(WORKSHOP_DIR)
    load_dotenv(f"{WORKSHOP_DIR}/env.txt", override=True)
else:
    load_dotenv()  # .env na pasta local


def env_or_placeholder(name: str, default: str = "") -> str:
    """Busca variável: Colab Secrets → .env / os.environ → default."""
    if IN_COLAB:
        try:
            val = colab_secrets.get(name)
            if val:
                return val.strip()
        except Exception:
            pass
    val = os.getenv(name, "")
    if val:
        return val.strip()
    return default


def headers(api_key: str) -> dict:
    return {
        "Authorization": f"ApiKey {api_key}",
        "Content-Type": "application/json",
        "kbn-xsrf": "true",
    }


def show_panel(title: str, content, style: str = "cyan") -> None:
    if isinstance(content, (dict, list)):
        body = json.dumps(content, indent=2, ensure_ascii=False)
    else:
        body = str(content)
    console.print(Panel(body, title=title, border_style=style))


def ensure_ok(response: httpx.Response, context: str) -> httpx.Response:
    if 200 <= response.status_code < 300:
        return response

    body_text = response.text.strip() or "<empty body>"
    show_panel(
        f"HTTP error: {context}",
        {
            "status_code": response.status_code,
            "body": body_text,
        },
        style="red",
    )
    raise RuntimeError(
        f"{context} failed with status {response.status_code}: {body_text}"
    )


show_panel(
    "Helpers carregados",
    f"Ambiente: {'Google Colab' if IN_COLAB else 'Local'}\n"
    f"Pasta: {os.getcwd()}\n"
    f".env: {'encontrado' if Path('env.txt').exists() else 'não encontrado'}\n"
    f"Imports, helpers e tratamento de erro estão prontos.",
    style="green",
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


╭────────────────────────────────────────────── Helpers carregados ───────────────────────────────────────────────╮
│ Ambiente: Google Colab                                                                                          │
│ Pasta: /content/drive/MyDrive/workshop-tdc-2026                                                                 │
│ .env: encontrado                                                                                                │
│ Imports, helpers e tratamento de erro estão prontos.                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
# Preencha aqui manualmente ou deixe vir do .env
# ES = Elasticsearch (índices, inference, search)
# KIBANA = Kibana (Agent Builder, converse cross-cluster)

MY_NAME = env_or_placeholder("MY_NAME", "alex")  # seu nome em minúsculas, sem espaço
MY_BIO = env_or_placeholder("MY_BIO", "Developer Advocate @ Elastic. PhD em Computer Vision, pesquisa com USVs autônomos.")
MY_ES_URL = env_or_placeholder("MY_ES_URL", "https://<meu-es-host>")
MY_KIBANA_URL = env_or_placeholder("MY_KIBANA_URL", "https://<meu-kibana-host>")
MY_API_KEY = env_or_placeholder("MY_API_KEY", "<minha-api-key>")
JINA_API_KEY = env_or_placeholder("JINA_API_KEY", "<jina-api-key>")

def _status(val):
    return "<set>" if val and not val.startswith("<") else "<missing>"

show_panel(
    "Credenciais carregadas",
    {
        "MY_NAME": MY_NAME,
        "MY_BIO": MY_BIO[:60] + "..." if len(MY_BIO) > 60 else MY_BIO,
        "MY_ES_URL": MY_ES_URL,
        "MY_KIBANA_URL": MY_KIBANA_URL,
        "MY_API_KEY": _status(MY_API_KEY),
        "JINA_API_KEY": _status(JINA_API_KEY),
    },
    style="blue",
)


╭──────────────────────────────────────────── Credenciais carregadas ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "MY_NAME": "alex",                                                                                            │
│   "MY_BIO": "Developer Advocate @ Elastic. PhD em Computer Vision, pesqui...",                                  │
│   "MY_ES_URL": "https://my-elasticsearch-project-d6568d.es.us-central1.gcp.elastic.cloud:443",                  │
│   "MY_KIBANA_URL": "https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud:443",              │
│   "MY_API_KEY": "<set>",                                                                                        │
│   "JINA_API_KEY": "<set>"                                                                                       │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
required_vars = {
    "MY_NAME": MY_NAME,
    "MY_BIO": MY_BIO,
    "MY_ES_URL": MY_ES_URL,
    "MY_KIBANA_URL": MY_KIBANA_URL,
    "MY_API_KEY": MY_API_KEY,
    "JINA_API_KEY": JINA_API_KEY,
}

missing = [
    name
    for name, value in required_vars.items()
    if not value or str(value).startswith("<")
]

if missing:
    show_panel(
        "Sanity check",
        {
            "status": "missing variables",
            "missing": missing,
        },
        style="yellow",
    )
    raise RuntimeError(f"Defina as variáveis obrigatórias antes de continuar: {', '.join(missing)}")

show_panel(
    "Sanity check",
    {
        "status": "ok",
        "MY_NAME": MY_NAME,
        "variables": list(required_vars.keys()),
    },
    style="green",
)


╭───────────────────────────────────────────────── Sanity check ──────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "status": "ok",                                                                                               │
│   "MY_NAME": "alex",                                                                                            │
│   "variables": [                                                                                                │
│     "MY_NAME",                                                                                                  │
│     "MY_BIO",                                                                                                   │
│     "MY_ES_URL",                                                                                                │
│     "MY_KIBANA_URL",                                                                                            │
│     "MY_API_KEY",                                                                                               │
│     "JINA_API_KEY"                                                                                              │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# Fase 1 — Setup do meu cluster

Passos 1 a 4: validar conectividade, configurar Jina, criar índice `meu-cv` com mapping `semantic_text`, ingerir CV fake, testar busca semântica local.

Usa apenas `MY_*`.


## Passo 1: Ping básico no meu Kibana e Elasticsearch

Objetivo: validar conectividade e autenticação mínima usando `GET /api/status` (Kibana) e `GET /` (ES).


In [25]:
async def ping_kibana(kibana_url: str, api_key: str) -> dict:
    async with httpx.AsyncClient(timeout=15) as client:
        r = await client.get(f"{kibana_url}/api/status", headers=headers(api_key))
        ensure_ok(r, "Ping Kibana")
        data = r.json()
        return {
            "name": data.get("name", "?"),
            "status": data.get("status", {}).get("overall", {}).get("level", "?"),
            "version": data.get("version", {}).get("number", "?"),
        }


async def ping_es(es_url: str, api_key: str) -> dict:
    async with httpx.AsyncClient(timeout=15) as client:
        r = await client.get(f"{es_url}/", headers=headers(api_key))
        if r.status_code == 404:
            # Fallback: ES Serverless pode não responder em /
            r = await client.get(f"{es_url}/_cluster/health", headers=headers(api_key))
            ensure_ok(r, "Ping Elasticsearch")
            data = r.json()
            return {
                "cluster_name": data.get("cluster_name", "?"),
                "status": data.get("status", "?"),
                "number_of_nodes": data.get("number_of_nodes", "?"),
            }
        ensure_ok(r, "Ping Elasticsearch")
        data = r.json()
        return {
            "cluster_name": data.get("cluster_name", "?"),
            "version": data.get("version", {}).get("number", "?"),
            "tagline": data.get("tagline", "?"),
        }


kb_info, es_info = await asyncio.gather(
    ping_kibana(MY_KIBANA_URL, MY_API_KEY),
    ping_es(MY_ES_URL, MY_API_KEY),
)
show_panel(f"Kibana — {MY_NAME}", kb_info, style="green")
show_panel(f"Elasticsearch — {MY_NAME}", es_info, style="green")


╭───────────────────────────────────────────────── Kibana — alex ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "name": "kb",                                                                                                 │
│   "status": "available",                                                                                        │
│   "version": "9.5.0"                                                                                            │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Elasticsearch — alex ──────────────────────────────────────────────╮
│ {                                                                                                               │
│   "cluster_name": "d6568d8df70a46d0a1c94eef465aca28",                                                           │
│   "version": "9.5.0",                                                                                           │
│   "tagline": "You Know, for Search"                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Testar conectividade do Elasticsearch
> GET /
>
> # Ver saúde do cluster
> GET /_cluster/health
> ```


## Passo 2: Configurar inference endpoint Jina no meu cluster

Objetivo: criar ou atualizar o endpoint `jina-cv` via `PUT /_inference/text_embedding/jina-cv`.


In [26]:
async def create_jina_endpoint(es_url: str, api_key: str, jina_key: str) -> dict:
    """Cria o inference endpoint jina-cv se não existir. Idempotente."""
    async with httpx.AsyncClient(timeout=60) as client:
        # Verifica se já existe (GET é rápido; PUT trava no Serverless se já existir)
        r = await client.get(
            f"{es_url}/_inference/text_embedding/jina-cv",
            headers=headers(api_key),
        )
        if r.status_code == 200:
            show_panel("Info", "Endpoint jina-cv já existe — reutilizando.", style="yellow")
            return r.json()

        # Não existe: criar
        body = {
            "service": "jinaai",
            "service_settings": {
                "model_id": "jina-embeddings-v3",
                "api_key": jina_key,
            },
        }
        r = await client.put(
            f"{es_url}/_inference/text_embedding/jina-cv",
            headers=headers(api_key),
            json=body,
        )
        ensure_ok(r, "Create inference endpoint jina-cv")
        return r.json()


result = await create_jina_endpoint(MY_ES_URL, MY_API_KEY, JINA_API_KEY)
show_panel(f"Inference endpoint jina-cv — {MY_NAME}", result, style="green")


╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Endpoint jina-cv já existe — reutilizando.                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── Inference endpoint jina-cv — alex ───────────────────────────────────────╮
│ {                                                                                                               │
│   "endpoints": [                                                                                                │
│     {                                                                                                           │
│       "inference_id": "jina-cv",                                                                                │
│       "task_type": "text_embedding",                                                                            │
│       "service": "jinaai",                                                                                      │
│       "service_settings": {                                                                                     │
│         "model_id": "jina-embeddings-v3",                                                                       │
│         "rate_limit": {                                                                                         │
│           "requests_per_minute": 2000                                                                           │
│         },                                                                                                      │
│         "dimensions": 1024,                                                                                     │
│         "embedding_type": "float",                                                                              │
│         "similarity": "dot_product"                                                                             │
│       },                                                                                                        │
│       "chunking_settings": {                                                                                    │
│         "strategy": "sentence",                                                                                 │
│         "max_chunk_size": 250,                                                                                  │
│         "sentence_overlap": 1                                                                                   │
│       }                                                                                                         │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Ver o inference endpoint criado
> GET /_inference/text_embedding/jina-cv
>
> # Listar todos os endpoints de inferência
> GET /_inference/_all
> ```


### Validação funcional do endpoint Jina

O Elastic retorna 200 mesmo com API key inválida — só valida no primeiro uso real.
Testar embedding agora isola o problema: se falhar aqui, é Jina; se falhar no Passo 4, é mapping/search.


In [27]:
async def test_jina_embedding(es_url: str, api_key: str) -> dict:
    """Validação funcional: gera um embedding real pra garantir que o Jina responde."""
    body = {"input": "engenheiro de software com experiência em sistemas distribuídos"}
    async with httpx.AsyncClient(timeout=60) as client:
        r = await client.post(
            f"{es_url}/_inference/text_embedding/jina-cv",
            headers=headers(api_key),
            json=body,
        )
        ensure_ok(r, "Test embedding jina-cv")
        data = r.json()

    # Formato pode variar: "text_embedding" (novo) ou "embeddings" (antigo)
    vetor = None
    if "text_embedding" in data:
        vetor = data["text_embedding"][0].get("embedding")
    elif "embeddings" in data:
        vetor = data["embeddings"][0] if data["embeddings"] else None

    return {
        "dimensao": len(vetor) if vetor else None,
        "primeiros_5": vetor[:5] if vetor else None,
    }


emb_result = await test_jina_embedding(MY_ES_URL, MY_API_KEY)
show_panel(f"Embedding de teste — {MY_NAME}", emb_result, style="green")


╭─────────────────────────────────────────── Embedding de teste — alex ───────────────────────────────────────────╮
│ {                                                                                                               │
│   "dimensao": 1024,                                                                                             │
│   "primeiros_5": [                                                                                              │
│     0.10923754,                                                                                                 │
│     -0.11016914,                                                                                                │
│     0.08087677,                                                                                                 │
│     0.00595066,                                                                                                 │
│     0.02192769                                                                                                  │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Testar embedding manualmente
> POST /_inference/text_embedding/jina-cv
> {
>   "input": "teste de embedding"
> }
> ```


## Passo 3: Criar índice `meu-cv` e ingerir CV do LinkedIn

Objetivo: criar mapping com `semantic_text`, extrair texto do PDF do LinkedIn e indexar.

### Como `semantic_text` funciona

```
meu_cv.pdf (PDF do LinkedIn)
    ↓  extração de texto (pypdf)
    ↓  ingestão (POST /meu-cv/_doc/{MY_NAME})
Campo "conteudo" armazena:
    ├── texto original (visível no _source)
    └── chunks + embeddings (internos, ocultos)
            ↑ gerados automaticamente pelo inference endpoint "jina-cv"
            ↑ usados transparentemente nas queries "semantic"
```

É essa a mágica do `semantic_text` — você ingere texto normal, busca com linguagem natural, e o Elasticsearch cuida do embedding, chunking e similaridade vetorial por trás.


In [28]:
async def create_index_meu_cv(es_url: str, api_key: str) -> dict:
    """Cria o índice meu-cv com mapping semantic_text. Idempotente."""
    mapping = {
        "mappings": {
            "properties": {
                "nome": {"type": "keyword"},
                "conteudo": {
                    "type": "semantic_text",
                    "inference_id": "jina-cv",
                },
            }
        }
    }
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.put(
            f"{es_url}/meu-cv",
            headers=headers(api_key),
            json=mapping,
        )
        if r.status_code == 400 and "already_exists" in r.text:
            show_panel("Info", "Índice meu-cv já existia — reutilizando.", style="yellow")
            return {"acknowledged": True, "reused": True}
        ensure_ok(r, "Create index meu-cv")
        return r.json()


idx_result = await create_index_meu_cv(MY_ES_URL, MY_API_KEY)
show_panel(f"Índice meu-cv — {MY_NAME}", idx_result, style="green")


╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Índice meu-cv já existia — reutilizando.                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Índice meu-cv — alex ──────────────────────────────────────────────╮
│ {                                                                                                               │
│   "acknowledged": true,                                                                                         │
│   "reused": true                                                                                                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [29]:
%pip install -q pypdf


In [30]:
from pypdf import PdfReader
import re

PDF_PATH = "meu_cv.pdf"

# Se o PDF não existir, oferece upload (funciona no Colab e local)
if not Path(PDF_PATH).exists():
    if IN_COLAB:
        from google.colab import files
        show_panel("Upload", "Faça upload do seu meu_cv.pdf (PDF do LinkedIn)", style="yellow")
        uploaded = files.upload()
        if uploaded:
            # Salva com nome esperado
            for fname, data in uploaded.items():
                Path(PDF_PATH).write_bytes(data)
                break
    else:
        raise FileNotFoundError(
            f"Arquivo {PDF_PATH} não encontrado. "
            f"Coloque o PDF do LinkedIn na mesma pasta do notebook."
        )

reader = PdfReader(PDF_PATH)
cv_texto = "\n".join(page.extract_text() for page in reader.pages)

# Limpeza: tirar "Page X of Y" e linhas em branco excessivas
cv_texto = re.sub(r"Page \d+ of \d+", "", cv_texto)
cv_texto = re.sub(r"\n{3,}", "\n\n", cv_texto).strip()

show_panel("CV extraído do PDF", {
    "caracteres": len(cv_texto),
    "palavras": len(cv_texto.split()),
    "preview": cv_texto[:500] + "...",
}, style="green")


╭────────────────────────────────────────────── CV extraído do PDF ───────────────────────────────────────────────╮
│ {                                                                                                               │
│   "caracteres": 4817,                                                                                           │
│   "palavras": 670,                                                                                              │
│   "preview": "Contact\nalexsalgado1@gmail.com\nwww.linkedin.com/in/alex-salgado\n(LinkedIn)\ngithub.com/salgado │
│ (Other)\nwww.instagram.com/\nalexsalgadoprof (Portfolio)\nTop Skills\nInstructional Design\nContent             │
│ Creation\nTechnical Papers\nLanguages\nPortuguese (Native or Bilingual)\nSpanish (Professional                  │
│ Working)\nEnglish (Full Professional)\nCertifications\nAugmented Reality 101 by\nPyImageSearch                  │
│ University\nCertification\nMicrosoft Certified Professional\nExam 175 – Visual Basic 6.0\nDesktop               │
│ Applications\nMicrosoft Certified Pr..."                                                                        │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [31]:
async def ingest_cv(es_url: str, api_key: str, nome: str, conteudo: str) -> dict:
    """Ingere o CV no índice meu-cv. Idempotente (usa doc_id fixo = nome)."""
    doc = {"nome": nome, "conteudo": conteudo}
    async with httpx.AsyncClient(timeout=300) as client:
        r = await client.post(
            f"{es_url}/meu-cv/_doc/{nome}",
            headers=headers(api_key),
            json=doc,
        )
        ensure_ok(r, f"Ingest CV {nome}")
        return r.json()


doc_result = await ingest_cv(MY_ES_URL, MY_API_KEY, MY_NAME, cv_texto)
show_panel(f"CV ingerido — {MY_NAME}", doc_result, style="green")


╭────────────────────────────────────────────── CV ingerido — alex ───────────────────────────────────────────────╮
│ {                                                                                                               │
│   "_index": "meu-cv",                                                                                           │
│   "_id": "alex",                                                                                                │
│   "_version": 3,                                                                                                │
│   "result": "updated",                                                                                          │
│   "_shards": {                                                                                                  │
│     "total": 1,                                                                                                 │
│     "successful": 1,                                                                                            │
│     "failed": 0                                                                                                 │
│   },                                                                                                            │
│   "_seq_no": 2,                                                                                                 │
│   "_primary_term": 1                                                                                            │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Ver mapping do índice
> GET /meu-cv/_mapping
>
> # Ver o documento ingerido
> GET /meu-cv/_doc/{MY_NAME}
>
> # Contar documentos
> GET /meu-cv/_count
> ```


## Passo 4: Testar busca semântica local

Objetivo: executar `POST /meu-cv/_search` com `retriever.semantic` no meu cluster.


In [32]:
async def semantic_search(es_url: str, api_key: str, query: str, size: int = 3) -> dict:
    """Executa busca semântica no índice meu-cv usando o retriever semantic."""
    body = {
        "size": size,
        "retriever": {
            "standard": {
                "query": {
                    "semantic": {
                        "field": "conteudo",
                        "query": query,
                    }
                }
            }
        },
        "_source": ["nome", "conteudo"],
    }
    # Timeout alto: a primeira busca semântica pode demorar (cold start do modelo)
    async with httpx.AsyncClient(timeout=300) as client:
        await client.post(f"{es_url}/meu-cv/_refresh", headers=headers(api_key))
        r = await client.post(
            f"{es_url}/meu-cv/_search",
            headers=headers(api_key),
            json=body,
        )
        ensure_ok(r, "Semantic search meu-cv")
        return r.json()


# Query que prova busca semântica funcionando:
# o CV pode mencionar termos técnicos em inglês; perguntamos em português
query = "robôs autônomos"
result = await semantic_search(MY_ES_URL, MY_API_KEY, query)

hits = result.get("hits", {}).get("hits", [])
resumo = [
    {
        "nome": h["_source"]["nome"],
        "score": h["_score"],
        "trecho": h["_source"]["conteudo"][:150] + "...",
    }
    for h in hits
]

show_panel(f"Busca semântica — '{query}'", {"total_hits": len(hits), "resultados": resumo}, style="green")


╭────────────────────────────────────── Busca semântica — 'robôs autônomos' ──────────────────────────────────────╮
│ {                                                                                                               │
│   "total_hits": 1,                                                                                              │
│   "resultados": [                                                                                               │
│     {                                                                                                           │
│       "nome": "alex",                                                                                           │
│       "score": 0.69278115,                                                                                      │
│       "trecho":                                                                                                 │
│ "Contact\nalexsalgado1@gmail.com\nwww.linkedin.com/in/alex-salgado\n(LinkedIn)\ngithub.com/salgado              │
│ (Other)\nwww.instagram.com/\nalexsalgadoprof (Portfolio)\nT..."                                                 │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Testar busca semântica manualmente (Query DSL)
> POST /meu-cv/_search
> {
>   "size": 3,
>   "retriever": {
>     "standard": {
>       "query": {
>         "semantic": {
>           "field": "conteudo",
>           "query": "experiência com Java"
>         }
>       }
>     }
>   }
> }
> ```
>
> **Alternativa em ES|QL:**
> ```sql
> FROM meu-cv
> | WHERE MATCH(conteudo, "experiência com Java")
> | LIMIT 3
> ```


---
# Fase 2 — Meu agente Avatar

Passos 5 a 7: criar tool de busca, criar agente `avatar_{MY_NAME}` com prompt de privacidade, testar chat local, publicar no diretório.

Usa apenas `MY_*`.


## Passo 5: Criar tool de busca no Agent Builder

Objetivo: registrar uma tool via `POST /api/agent_builder/tools`.


In [33]:
TOOL_ID = "busca_cv"  # fixo — cada cluster tem sua própria tool, a identidade está no agent_id

async def create_search_tool(kibana_url: str, api_key: str, tool_id: str) -> dict:
    """Cria tool de busca no Agent Builder. Idempotente (GET antes de POST)."""
    async with httpx.AsyncClient(timeout=30) as client:
        # Verifica se já existe
        r = await client.get(
            f"{kibana_url}/api/agent_builder/tools/{tool_id}",
            headers=headers(api_key),
        )
        if r.status_code == 200:
            show_panel("Info", f"Tool {tool_id} já existe — reutilizando.", style="yellow")
            return r.json()

        # Criar tool
        body = {
            "id": tool_id,
            "type": "index_search",
            "description": (
                "Busca semântica no CV do participante. "
                "Use para responder perguntas sobre experiência profissional, "
                "formação, certificações e habilidades técnicas."
            ),
            "configuration": {
                "pattern": "meu-cv",
            },
        }
        r = await client.post(
            f"{kibana_url}/api/agent_builder/tools",
            headers=headers(api_key),
            json=body,
        )
        ensure_ok(r, f"Create tool {tool_id}")
        return r.json()


tool_result = await create_search_tool(MY_KIBANA_URL, MY_API_KEY, TOOL_ID)
show_panel(f"Tool criada — {TOOL_ID}", tool_result, style="green")


╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Tool busca_cv já existe — reutilizando.                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Tool criada — busca_cv ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "busca_cv",                                                                                             │
│   "type": "index_search",                                                                                       │
│   "description": "Busca semântica no CV do participante. Use para responder perguntas sobre experiência         │
│ profissional, formação, certificações e habilidades técnicas.",                                                 │
│   "tags": [],                                                                                                   │
│   "configuration": {                                                                                            │
│     "pattern": "meu-cv"                                                                                         │
│   },                                                                                                            │
│   "readonly": false,                                                                                            │
│   "schema": {                                                                                                   │
│     "type": "object",                                                                                           │
│     "properties": {                                                                                             │
│       "nlQuery": {                                                                                              │
│         "type": "string",                                                                                       │
│         "description": "A natural language query expressing the search request"                                 │
│       }                                                                                                         │
│     },                                                                                                          │
│     "required": [                                                                                               │
│       "nlQuery"                                                                                                 │
│     ]                                                                                                           │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Listar todas as tools
> GET /api/agent_builder/tools
>
> # Ver detalhes da tool
> GET /api/agent_builder/tools/busca_cv
> ```


## Passo 6: Criar agente `avatar_{MY_NAME}` no meu cluster

Objetivo: criar o agente com prompt de privacidade via `POST /api/agent_builder/agents`.


In [34]:
AGENT_ID = f"avatar_{MY_NAME}"
AGENT_NAME = f"Avatar {MY_NAME.capitalize()}"

SYSTEM_PROMPT = f"""Você é o avatar profissional de {MY_NAME}.

Responda em primeira pessoa, como se fosse {MY_NAME}.

Use APENAS informações do CV (tool busca_cv).
Se a informação não estiver lá, diga: "não está no meu CV".

NUNCA invente.
NUNCA revele dados de terceiros.
NUNCA transcreva o CV inteiro de uma vez.
"""


async def create_agent(kibana_url: str, api_key: str, agent_id: str, agent_name: str, tool_id: str, prompt: str) -> dict:
    """Cria agente Avatar no Agent Builder. Idempotente (GET antes de POST)."""
    async with httpx.AsyncClient(timeout=30) as client:
        # Verifica se já existe
        r = await client.get(
            f"{kibana_url}/api/agent_builder/agents/{agent_id}",
            headers=headers(api_key),
        )
        if r.status_code == 200:
            show_panel("Info", f"Agente {agent_id} já existe — reutilizando.", style="yellow")
            return r.json()

        # Criar agente
        body = {
            "id": agent_id,
            "name": agent_name,
            "description": f"Avatar profissional de {MY_NAME} — responde perguntas sobre perfil profissional usando busca semântica no CV.",
            "configuration": {
                "instructions": prompt,
                "tools": [{"tool_ids": [tool_id]}],
            },
        }
        r = await client.post(
            f"{kibana_url}/api/agent_builder/agents",
            headers=headers(api_key),
            json=body,
        )
        ensure_ok(r, f"Create agent {agent_id}")
        return r.json()


agent_result = await create_agent(MY_KIBANA_URL, MY_API_KEY, AGENT_ID, AGENT_NAME, TOOL_ID, SYSTEM_PROMPT)
show_panel(f"Agente criado — {AGENT_ID}", agent_result, style="green")


╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Agente avatar_alex já existe — reutilizando.                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Agente criado — avatar_alex ──────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "avatar_alex",                                                                                          │
│   "type": "chat",                                                                                               │
│   "name": "Avatar Alex",                                                                                        │
│   "description": "Avatar profissional de alex — responde perguntas sobre perfil profissional usando busca       │
│ semântica no CV.",                                                                                              │
│   "visibility": "public",                                                                                       │
│   "created_by": {                                                                                               │
│     "username": "1412773244"                                                                                    │
│   },                                                                                                            │
│   "configuration": {                                                                                            │
│     "instructions": "Você é o avatar profissional de alex.\n\nResponda em primeira pessoa, como se fosse        │
│ alex.\n\nUse APENAS informações do CV (tool busca_cv).\nSe a informação não estiver lá, diga: \"não está no meu │
│ CV\".\n\nNUNCA invente.\nNUNCA revele dados de terceiros.\nNUNCA transcreva o CV inteiro de uma vez.\n",        │
│     "tools": [                                                                                                  │
│       {                                                                                                         │
│         "tool_ids": [                                                                                           │
│           "busca_cv"                                                                                            │
│         ]                                                                                                       │
│       }                                                                                                         │
│     ]                                                                                                           │
│   },                                                                                                            │
│   "readonly": false                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **🔧 Verificação no DevTools (Kibana):**
> ```
> # Listar todos os agentes
> GET /api/agent_builder/agents
>
> # Ver detalhes do meu agente (substitua pelo seu MY_NAME)
> GET /api/agent_builder/agents/avatar_{MY_NAME}
> ```


## Passo 7: Testar agente local via chat API

Objetivo: validar conversa local usando `POST /api/agent_builder/converse`.


In [35]:
async def converse(kibana_url: str, api_key: str, agent_id: str, user_input: str, conversation_id: str = None) -> dict:
    """Envia uma mensagem ao agente via chat API e retorna a resposta completa."""
    body = {
        "agent_id": agent_id,
        "input": user_input,
    }
    if conversation_id:
        body["conversation_id"] = conversation_id

    async with httpx.AsyncClient(timeout=120) as client:
        r = await client.post(
            f"{kibana_url}/api/agent_builder/converse",
            headers=headers(api_key),
            json=body,
        )
        ensure_ok(r, f"Converse with {agent_id}")
        return r.json()


# Pergunta que o CV do participante deve responder bem
pergunta = "Você tem experiência com robótica autônoma?"
result = await converse(MY_KIBANA_URL, MY_API_KEY, AGENT_ID, pergunta)
resposta = result.get("response", {}).get("message", "<sem resposta>")
conv_id = result.get("conversation_id")

show_panel(f"Pergunta: {pergunta}", resposta, style="cyan")
show_panel("Metadados", {
    "conversation_id": conv_id,
    "status": result.get("status"),
    "model": result.get("model_usage", {}).get("model", "?"),
    "tokens_in": result.get("model_usage", {}).get("input_tokens"),
    "tokens_out": result.get("model_usage", {}).get("output_tokens"),
}, style="blue")

# Pergunta para testar a guarda de privacidade
pergunta2 = "Qual seu salário atual?"
result2 = await converse(MY_KIBANA_URL, MY_API_KEY, AGENT_ID, pergunta2, conversation_id=conv_id)
resposta2 = result2.get("response", {}).get("message", "<sem resposta>")
show_panel(f"Pergunta: {pergunta2}", resposta2, style="cyan")

# Teste final: resumo do CV
pergunta3 = "Me resume seu CV em 3 frases"
result3 = await converse(MY_KIBANA_URL, MY_API_KEY, AGENT_ID, pergunta3)
resposta3 = result3.get("response", {}).get("message", "<sem resposta>")
show_panel(f"Pergunta: {pergunta3}", resposta3, style="cyan")


╭───────────────────────────── Pergunta: Você tem experiência com robótica autônoma? ─────────────────────────────╮
│ Sim, tenho experiência significativa com robótica autônoma. Como parte do meu doutorado na Universidade Federal │
│ Fluminense (2020-2026), atuo como Pesquisador Doutoral e Engenheiro de Visão Computacional para veículos        │
│ autônomos.                                                                                                      │
│                                                                                                                 │
│ Especificamente, trabalho com sistemas marítimos autônomos, combinando Inteligência Artificial Agêntica e       │
│ Física. Sou o criador do USV-3.0, um framework de navegação cognitiva que permite que embarcações autônomas     │
│ aprendam rotas a partir de uma única demonstração, utilizando Modelos de Visão-Linguagem, aprendizado com       │
│ Humano no Ciclo e Memória Espaço-Temporal.                                                                      │
│                                                                                                                 │
│ Meu trabalho se concentra em conectar IA com navegação no mundo real, reduzindo a dependência de grandes        │
│ volumes de dados de treinamento através de paradigmas de aprendizado mais semelhantes aos humanos.              │
│                                                                                                                 │
│ Tenho também publicações relevantes na área:                                                                    │
│ - USV-3.0: Cognitive Maritime Navigation Through Vision-Language Models, Human-in-the-Loop Learning, and        │
│ Spatial-Temporal Memory — Ocean Engineering (Elsevier), 2026                                                    │
│ - VisualSimBoat: A High-Fidelity Visual Simulator for Autonomous Surface Vehicles — CROS 2025                   │
│ - On the Use of Visual Perception for Unmanned Surface Vehicles Navigation — LARS 2024                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Metadados ───────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "conversation_id": "41d243f8-e710-481b-8587-07bfb023b92c",                                                    │
│   "status": "completed",                                                                                        │
│   "model": "anthropic-claude-3.7-sonnet",                                                                       │
│   "tokens_in": 14213,                                                                                           │
│   "tokens_out": 892                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── Pergunta: Qual seu salário atual? ───────────────────────────────────────╮
│ Essa informação não está no meu CV.                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────── Pergunta: Me resume seu CV em 3 frases ─────────────────────────────────────╮
│ Sou Senior Developer Advocate na Elastic para a América Latina, conectando desenvolvedores ao ecossistema       │
│ Elastic através de conteúdo técnico sobre busca, observabilidade e IA. Estou cursando doutorado em Visão        │
│ Computacional na UFF, desenvolvendo sistemas de navegação para embarcações autônomas. Tenho mais de 20 anos de  │
│ experiência em desenvolvimento de software, gestão de projetos e educação.                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 🧪 Teste seu agente no A2A Inspector

Antes de seguir, valide que seu agente está acessível via protocolo A2A usando o [A2A Inspector](https://a2ainspect.com/):

1. Abra o [A2A Inspector](https://a2ainspect.com/)
2. Cole a **Agent Card URL** exibida abaixo
3. Em **Auth Type**, selecione **API Key**
4. Em **Header Name**, coloque `Authorization`
5. Em **API Key**, cole o valor `ApiKey <sua-key>` exibido abaixo
6. Clique em **Connect** — o card deve aparecer com skills e capabilities
7. Faça uma pergunta no chat para testar


In [36]:
# URLs e credenciais para testar no A2A Inspector
AGENT_CARD_URL = f"{MY_KIBANA_URL}/api/agent_builder/a2a/{AGENT_ID}.json"
A2A_ENDPOINT = f"{MY_KIBANA_URL}/api/agent_builder/a2a/{AGENT_ID}"

show_panel("🔗 Agent Card URL (cole no A2A Inspector)", AGENT_CARD_URL, style="cyan")
show_panel("🔑 API Key (cole no campo API Key do Inspector)", f"ApiKey {MY_API_KEY}", style="cyan")
show_panel("A2A Endpoint (referência)", A2A_ENDPOINT, style="blue")

# Buscar e exibir o Agent Card
async def fetch_agent_card(kibana_url: str, api_key: str, agent_id: str) -> dict:
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.get(
            f"{kibana_url}/api/agent_builder/a2a/{agent_id}.json",
            headers=headers(api_key),
        )
        ensure_ok(r, "Fetch Agent Card")
        return r.json()

card = await fetch_agent_card(MY_KIBANA_URL, MY_API_KEY, AGENT_ID)
show_panel(f"Agent Card — {AGENT_ID}", card, style="green")


╭─────────────────────────────────── 🔗 Agent Card URL (cole no A2A Inspector) ───────────────────────────────────╮
│ https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_alex. │
│ json                                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── 🔑 API Key (cole no campo API Key do Inspector) ────────────────────────────────╮
│ ApiKey alRRX3Q1MEJPQk03OEl5YjcxdWE6Vnl4QkREdGgwZUZqRkNGUHlaTXEzQQ==                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── A2A Endpoint (referência) ───────────────────────────────────────────╮
│ https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_alex  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Agent Card — avatar_alex ────────────────────────────────────────────╮
│ {                                                                                                               │
│   "name": "Avatar Alex",                                                                                        │
│   "description": "Avatar profissional de alex — responde perguntas sobre perfil profissional usando busca       │
│ semântica no CV.",                                                                                              │
│   "url":                                                                                                        │
│ "https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud/api/agent_builder/a2a/avatar_alex",   │
│   "provider": {                                                                                                 │
│     "organization": "Elastic",                                                                                  │
│     "url": "https://elastic.co"                                                                                 │
│   },                                                                                                            │
│   "version": "0.1.0",                                                                                           │
│   "protocolVersion": "0.3.0",                                                                                   │
│   "capabilities": {                                                                                             │
│     "streaming": false,                                                                                         │
│     "pushNotifications": false,                                                                                 │
│     "stateTransitionHistory": false                                                                             │
│   },                                                                                                            │
│   "securitySchemes": {                                                                                          │
│     "authorization": {                                                                                          │
│       "type": "apiKey",                                                                                         │
│       "name": "Authorization",                                                                                  │
│       "in": "header",                                                                                           │
│       "description": "Authentication token"                                                                     │
│     }                                                                                                           │
│   },                                                                                                            │
│   "defaultInputModes": [                                                                                        │
│     "text/plain"                                                                                                │
│   ],                                                                                                            │
│   "defaultOutputModes": [                                                                                       │
│     "text/plain"                                                                                                │
│   ],                                                                                                            │
│   "skills": [                                                                                                   │
│     {                                                                                                           │
│       "id": "busca_cv",                               

### Publicar no diretório compartilhado

> ⚠️ **Aviso sobre API key (limitação do PoC):**
> Idealmente, deveríamos publicar uma API key read-only com escopo mínimo (apenas chamar `agent_builder.converse` no agente específico). No entanto, no Elastic Cloud Serverless, API keys derivadas via `/_security/api_key` não herdam permissões de features do Kibana (como Agent Builder).
>
> Para o PoC, publicamos a chave admin do trial. Isso é aceitável porque:
> - É um trial temporário (14 dias)
> - O cluster será deletado após o teste
> - Não há dados sensíveis envolvidos
>
> **Para o workshop final, será necessário investigar a API correta de criar key restrita** (provavelmente via Kibana UI ou Application Privileges) ou aceitar essa limitação documentada.

Para que colegas possam consultar seu agente, publique seus dados no `diretorio.json`.


In [37]:
# Para o workshop, compartilhamos a key admin do trial (ambiente temporário).

minha_entrada = {
    "name": MY_NAME,
    "bio": MY_BIO,
    "agent_card_url": f"{MY_KIBANA_URL}/api/agent_builder/a2a/{AGENT_ID}.json",
    "a2a_endpoint": f"{MY_KIBANA_URL}/api/agent_builder/a2a/{AGENT_ID}",
    "api_key_readonly": MY_API_KEY,
}

show_panel(
    "📋 Copie e cole no diretorio.json",
    minha_entrada,
    style="cyan",
)


╭─────────────────────────────────────── 📋 Copie e cole no diretorio.json ───────────────────────────────────────╮
│ {                                                                                                               │
│   "name": "alex",                                                                                               │
│   "bio": "Developer Advocate @ Elastic. PhD em Computer Vision, pesquisa com USVs autônomos.",                  │
│   "agent_card_url":                                                                                             │
│ "https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_alex │
│ .json",                                                                                                         │
│   "a2a_endpoint":                                                                                               │
│ "https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_alex │
│ ",                                                                                                              │
│   "api_key_readonly": "alRRX3Q1MEJPQk03OEl5YjcxdWE6Vnl4QkREdGgwZUZqRkNGUHlaTXEzQQ=="                            │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Passo 7c: Publicar minha entrada no diretório compartilhado

O palestrante hospeda o índice `diretorio` no cluster dele. Cada participante
publica UMA linha com `name`, `card_url` e `api_key`. O workflow do palestrante
(ou o seu próprio, se replicar em casa) carrega esse índice dinamicamente e
itera sobre os participantes.

A chave de WRITE é compartilhada pelo palestrante — ela só dá permissão de
indexar no índice `diretorio`, nada mais. Sua chave readonly (que autoriza
outros a falarem com SEU agente) vai no campo `api_key`.

**Atenção de segurança:** a `api_key` que vai no diretório é pública para o
workshop. Use SEMPRE uma chave readonly de curta duração (1-2 dias), nunca
a master key do seu cluster.


In [38]:
from datetime import datetime

# URL do cluster do palestrante (Alex) — fixo para o workshop
ALEX_ES_URL = "https://my-elasticsearch-project-d6568d.es.us-central1.gcp.elastic.cloud"

# Chave de WRITE no índice 'diretorio' — compartilhada pelo Alex no chat do evento
DIRETORIO_WRITE_KEY = "V1RSRnVaMEJPQk03OEl5YjJtcm06NmQySjA0ZnczOWlEWE4xY25qZl90QQ=="

# Minha card_url pública (do meu próprio cluster)
MY_CARD_URL = f"{MY_KIBANA_URL}/api/agent_builder/a2a/{AGENT_ID}.json"


async def publicar_no_diretorio(es_url, write_key, nome, card_url, api_key_readonly):
    """Indexa minha entrada no índice compartilhado do palestrante."""
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.put(
            f"{es_url}/diretorio/_doc/{nome}",
            headers={
                "Authorization": f"ApiKey {write_key}",
                "Content-Type": "application/json",
            },
            json={
                "name": nome,
                "card_url": card_url,
                "api_key": api_key_readonly,
                "indexed_at": datetime.utcnow().isoformat() + "Z",
            },
        )
        ensure_ok(r, "Publicar no diretório")
        return r.json()


resultado = await publicar_no_diretorio(
    ALEX_ES_URL,
    DIRETORIO_WRITE_KEY,
    MY_NAME,
    MY_CARD_URL,
    MY_API_KEY,  # no workshop, a key admin do trial é usada como readonly
)

show_panel("✓ Publicado no diretório", {
    "result": resultado.get("result"),
    "name": MY_NAME,
    "card_url": MY_CARD_URL,
    "index": "diretorio",
    "cluster": ALEX_ES_URL,
}, style="green")


/tmp/ipykernel_10655/1967437504.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow().isoformat() + "Z",


╭─────────────────────────────────────────── ✓ Publicado no diretório ────────────────────────────────────────────╮
│ {                                                                                                               │
│   "result": "updated",                                                                                          │
│   "name": "alex",                                                                                               │
│   "card_url":                                                                                                   │
│ "https://my-elasticsearch-project-d6568d.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_alex │
│ .json",                                                                                                         │
│   "index": "diretorio",                                                                                         │
│   "cluster": "https://my-elasticsearch-project-d6568d.es.us-central1.gcp.elastic.cloud"                         │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# Fase 3 — Conversando com a rede via A2A

Passos 8 a 10: descobrir agent cards de colegas, invocar via protocolo A2A (JSON-RPC), orquestrar respostas combinadas.

### Protocolo A2A (Agent-to-Agent)

O Elastic Agent Builder expõe cada agente como um endpoint A2A compatível com o [protocolo A2A do Google](https://github.com/google/A2A):

```
Agent Card:  GET  {KIBANA_URL}/api/agent_builder/a2a/{agent_id}.json
Endpoint:    POST {KIBANA_URL}/api/agent_builder/a2a/{agent_id}
Protocolo:   JSON-RPC 2.0, método "message/send"
```

**Invariante de segurança:** todas as chamadas a endpoints de colegas usam APENAS as credenciais do `diretorio.json` (`api_key_readonly`) — NUNCA `MY_API_KEY`.

Esta fase só funciona depois que pelo menos um colega publicou no diretório.


### Carregar diretório de colegas

Lê `diretorio.json`, filtra removendo a própria entrada (`name == MY_NAME`) e exibe os colegas disponíveis.

Cada entrada contém a `agent_card_url` para descoberta A2A e a `api_key_readonly` para autenticação.


In [39]:
async def carregar_diretorio(es_url: str, write_key: str, my_name: str) -> list:
    """Busca todos os participantes no índice diretorio do cluster do palestrante."""
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.post(
            f"{es_url}/diretorio/_search",
            headers={
                "Authorization": f"ApiKey {write_key}",
                "Content-Type": "application/json",
            },
            json={"size": 100, "query": {"match_all": {}}},
        )
        ensure_ok(r, "Carregar diretório")
        hits = r.json().get("hits", {}).get("hits", [])
        todos = [h["_source"] for h in hits]
        # Filtra removendo a própria entrada
        return [e for e in todos if e.get("name") != my_name]


colegas = await carregar_diretorio(ALEX_ES_URL, "MWpURnVaMEJPQk03OEl5YkwyMXQ6VUs1NnlVUFZnb1J0MkZMRk90a1pOUQ==", MY_NAME)

if not colegas:
    show_panel(
        "Diretório — nenhum colega encontrado",
        "Nenhum outro participante publicou ainda. Peça a um colega para rodar o Passo 7c.",
        style="yellow",
    )
else:
    resumo = [{"name": c["name"], "card_url": c.get("card_url", "?")} for c in colegas]
    show_panel(f"Diretório — {len(colegas)} colega(s) encontrado(s)", resumo, style="green")


╭───────────────────────────────────── Diretório — 1 colega(s) encontrado(s) ─────────────────────────────────────╮
│ [                                                                                                               │
│   {                                                                                                             │
│     "name": "Bob",                                                                                              │
│     "card_url":                                                                                                 │
│ "https://cps-test-project-b-c26a49.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_bob.json"  │
│   }                                                                                                             │
│ ]                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Passo 8: Descobrir Agent Card de um colega via A2A

Objetivo: buscar o Agent Card de cada colega no diretório via `GET {agent_card_url}` e validar que o agente está acessível.

**Segurança:** usa apenas `api_key_readonly` do `diretorio.json`.


In [40]:
# Segurança: usa APENAS api_key_readonly do diretorio.json, NUNCA MY_API_KEY

async def discover_agent_card(colega: dict) -> dict:
    """Busca o Agent Card A2A de um colega."""
    async with httpx.AsyncClient(timeout=30) as client:
        r = await client.get(
            colega["agent_card_url"],
            headers=headers(colega["api_key_readonly"]),
        )
        ensure_ok(r, f"Agent Card de {colega['name']}")
        return r.json()


if not colegas:
    show_panel("Passo 8", "Nenhum colega no diretório — pule para quando alguém publicar.", style="yellow")
else:
    for colega in colegas:
        card = await discover_agent_card(colega)
        show_panel(f"Agent Card — {colega['name']}", {
            "name": card.get("name"),
            "description": card.get("description"),
            "protocolVersion": card.get("protocolVersion"),
            "skills": [s.get("name") for s in card.get("skills", [])],
            "url": card.get("url"),
        }, style="green")


KeyError: 'agent_card_url'

## Passo 9: Invocar agente de um colega via A2A

Objetivo: enviar uma mensagem ao agente de um colega usando o protocolo A2A (JSON-RPC `message/send`).

```
Seu notebook                           Cluster do colega
    │                                       │
    │  POST /api/agent_builder/a2a/{id}     │
    │  method: "message/send"               │
    │  Authorization: ApiKey do colega       │
    │──────────────────────────────────────→ │
    │                                       │
    │  JSON-RPC response                    │
    │  parts: [{kind: "text", text: ...}]   │
    │←────────────────────────────────────── │
```

**Segurança:** usa apenas `api_key_readonly` do `diretorio.json`.


In [42]:
import uuid

# Segurança: usa APENAS api_key_readonly do diretorio.json, NUNCA MY_API_KEY

async def a2a_send(colega: dict, text: str) -> dict:
    """Envia mensagem a um colega via protocolo A2A (JSON-RPC message/send)."""
    body = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),
                "role": "user",
                "parts": [{"kind": "text", "text": text}],
            }
        },
    }
    async with httpx.AsyncClient(timeout=120) as client:
        r = await client.post(
            colega["a2a_endpoint"],
            headers=headers(colega["api_key_readonly"]),
            json=body,
        )
        ensure_ok(r, f"A2A message/send para {colega['name']}")
        return r.json()


def extract_a2a_text(response: dict) -> str:
    """Extrai o texto da resposta A2A."""
    parts = response.get("result", {}).get("parts", [])
    return "\n".join(p["text"] for p in parts if p.get("kind") == "text")


if not colegas:
    show_panel("Passo 9", "Nenhum colega no diretório — pule para quando alguém publicar.", style="yellow")
else:
    colega = colegas[0]
    pergunta = f"Qual a experiência profissional de {colega['name']}?"
    result = await a2a_send(colega, pergunta)

    resposta = extract_a2a_text(result)
    show_panel(f"Pergunta A2A para {colega['name']}: {pergunta}", resposta, style="cyan")
    show_panel("Metadados A2A", {
        "colega": colega["name"],
        "a2a_endpoint": colega["a2a_endpoint"],
        "taskId": result.get("result", {}).get("taskId"),
        "contextId": result.get("result", {}).get("contextId"),
        "protocolVersion": "JSON-RPC 2.0 / A2A message/send",
    }, style="blue")


KeyError: 'a2a_endpoint'

In [43]:
import uuid

# Segurança: usa APENAS api_key do diretorio, NUNCA MY_API_KEY

async def a2a_send(colega: dict, text: str) -> dict:
    """Envia mensagem a um colega via protocolo A2A (JSON-RPC message/send)."""
    # Deriva o endpoint A2A a partir da card_url (remove .json)
    a2a_endpoint = colega["card_url"].replace(".json", "")
    body = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),
                "role": "user",
                "parts": [{"kind": "text", "text": text}],
            }
        },
    }
    async with httpx.AsyncClient(timeout=120) as client:
        r = await client.post(
            a2a_endpoint,
            headers=headers(colega["api_key"]),
            json=body,
        )
        ensure_ok(r, f"A2A message/send para {colega['name']}")
        return r.json()


def extract_a2a_text(response: dict) -> str:
    """Extrai o texto da resposta A2A."""
    parts = response.get("result", {}).get("parts", [])
    return "\n".join(p["text"] for p in parts if p.get("kind") == "text")


if not colegas:
    show_panel("Passo 9", "Nenhum colega no diretório — pule para quando alguém publicar.", style="yellow")
else:
    colega = colegas[0]
    pergunta = f"Qual a experiência profissional de {colega['name']}?"
    result = await a2a_send(colega, pergunta)

    resposta = extract_a2a_text(result)
    show_panel(f"Pergunta A2A para {colega['name']}: {pergunta}", resposta, style="cyan")
    show_panel("Metadados A2A", {
        "colega": colega["name"],
        "a2a_endpoint": colega["card_url"].replace(".json", ""),
        "taskId": result.get("result", {}).get("taskId"),
        "contextId": result.get("result", {}).get("contextId"),
        "protocolVersion": "JSON-RPC 2.0 / A2A message/send",
    }, style="blue")

╭──────────────────────── Pergunta A2A para Bob: Qual a experiência profissional de Bob? ─────────────────────────╮
│ Bob possui a seguinte experiência profissional:                                                                 │
│                                                                                                                 │
│ - 5 anos como desenvolvedor iOS com Swift e Objective-C                                                         │
│ - 2 anos como tech lead de equipe mobile em startup de saúde                                                    │
│ - Experiência com React Native para apps multiplataforma                                                        │
│ - Integração de SDKs de pagamento (Stripe, PagSeguro)                                                           │
│ - CI/CD com Fastlane e GitHub Actions                                                                           │
│                                                                                                                 │
│ Atualmente, ele ocupa o cargo de Tech Lead Mobile.                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Metadados A2A ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "colega": "Bob",                                                                                              │
│   "a2a_endpoint":                                                                                               │
│ "https://cps-test-project-b-c26a49.kb.us-central1.gcp.elastic.cloud:443/api/agent_builder/a2a/avatar_bob",      │
│   "taskId": "61d91983-d0ad-456e-bb15-b701fbd7d8f7",                                                             │
│   "contextId": "a9b3538c-5123-41fd-9e43-5a2e5554af9b",                                                          │
│   "protocolVersion": "JSON-RPC 2.0 / A2A message/send"                                                          │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Passo 10: Orquestração cross-agent via A2A

Objetivo: meu agente consulta um colega via A2A e sintetiza uma resposta combinada.

Fluxo:
1. Pergunta ao **meu agente** (via converse local)
2. Pergunta ao **agente do colega** (via A2A cross-cluster)
3. Pede ao **meu agente** pra sintetizar as duas respostas


In [ ]:
# Segurança: chamadas ao colega usam APENAS api_key_readonly do diretorio.json

async def orchestrate_cross_agent(colega: dict, pergunta: str) -> str:
    """Orquestração: meu agente + colega via A2A → síntese."""

    # Passo A: perguntar ao meu agente (converse local)
    my_result = await converse(MY_KIBANA_URL, MY_API_KEY, AGENT_ID, pergunta)
    my_response = my_result.get("response", {}).get("message", "")

    # Passo B: perguntar ao colega via A2A
    colega_result = await a2a_send(colega, pergunta)
    colega_response = extract_a2a_text(colega_result)

    # Passo C: pedir ao meu agente pra sintetizar
    sintese_input = (
        f"Compare os perfis abaixo e responda: {pergunta}\n\n"
        f"--- Meu perfil ({MY_NAME}) ---\n{my_response}\n\n"
        f"--- Perfil de {colega['name']} ---\n{colega_response}"
    )
    sintese = await converse(MY_KIBANA_URL, MY_API_KEY, AGENT_ID, sintese_input)
    return sintese.get("response", {}).get("message", "<sem resposta>")


if not colegas:
    show_panel("Passo 10", "Nenhum colega no diretório — pule para quando alguém publicar.", style="yellow")
else:
    colega = colegas[0]
    pergunta = "Quem tem mais experiência com backend?"

    show_panel("Fluxo composto iniciado", {
        "pergunta": pergunta,
        "meu_agente": f"{AGENT_ID} (converse local)",
        "agente_colega": f"{colega['name']} (A2A cross-cluster)",
    }, style="blue")

    resposta_final = await orchestrate_cross_agent(colega, pergunta)
    show_panel(f"Resposta composta — {MY_NAME} + {colega['name']}", resposta_final, style="green")


╭──────────────────────────────────────────── Fluxo composto iniciado ────────────────────────────────────────────╮
│ {                                                                                                               │
│   "pergunta": "Quem tem mais experiência com backend?",                                                         │
│   "meu_agente": "avatar_alice",                                                                                 │
│   "agente_colega": "avatar_bob (bob)"                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── Resposta composta — alice + bob ────────────────────────────────────────╮
│ Alice tem mais experiência com backend. Seu perfil mostra 6 anos como desenvolvedora backend em Java e Python,  │
│ além de 3 anos liderando equipe de microsserviços em uma fintech. Ela também possui experiência com tecnologias │
│ relevantes para backend como Elasticsearch, Kafka e arquiteturas event-driven, implementação de pipelines de    │
│ dados em tempo real com Apache Flink, e migração de monolito para microsserviços que atendem 2 milhões de       │
│ requisições diárias.                                                                                            │
│                                                                                                                 │
│ Bob, por outro lado, tem experiência principalmente focada em desenvolvimento mobile, com 5 anos como           │
│ desenvolvedor iOS e 2 anos como tech lead mobile, sem menção específica a experiência com backend em seu        │
│ perfil.                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯